In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "bank_dev")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("storageName", "saprojectbankdeveastus")
dbutils.widgets.text("containerSilver", "silver")

In [0]:
catalogo         = dbutils.widgets.get("catalogo")
esquema_bronze   = dbutils.widgets.get("esquema_bronze")
esquema_silver   = dbutils.widgets.get("esquema_silver")
storageName      = dbutils.widgets.get("storageName")
containerSilver  = dbutils.widgets.get("containerSilver")

In [0]:
checkpoint_location_s = f"abfss://{containerSilver}@{storageName}.dfs.core.windows.net/_checkpoints/cards" 


tabla_bronze_cards = f"{catalogo}.{esquema_bronze}.cards"
tabla_silver_cards = f"{catalogo}.{esquema_silver}.dim_cards"

In [0]:
_df = spark.readStream.table("bank_dev.bronze.cards")

In [0]:
# ===================== TRANSFORMACIÓN CARDS =====================

df_silver_cards = (
    _df 
    # 1. Casteos Num y Booleanos
    .withColumn("credit_limit", F.col("credit_limit").cast("decimal(15,2)")) \
    .withColumn("has_chip", F.when(F.col("has_chip") == "YES", True).otherwise(False)) \
    .withColumn("card_on_dark_web", F.when(F.col("card_on_dark_web") == "YES", True).otherwise(False)) \

    # Usando tu lógica segura y llevándolo al fin de mes real de expiración:
    .withColumn("expires", F.last_day(F.to_date(F.concat(F.col("expires"), F.lit("/01")), "MM/yyyy/dd")))
    .withColumn("acct_open_date", F.to_date(F.col("acct_open_date"), "MM/yyyy")) \

    # 3. Seguridad: Enmascaramiento de tarjeta (dejando solo últimos 4 dig)
    .withColumn("card_number_masked", F.concat(F.lit("****-****-****-"), F.substring(F.col("card_number"), -4, 4))) \

    # 4. Columnas de Control SCD2
    # .withColumn("valid_from", F.current_timestamp()) \
    .withColumn("valid_from", F.to_timestamp(F.lit("1900-01-01 00:00:00")))
    .withColumn("valid_to", F.to_timestamp(F.lit("9999-12-31 23:59:59"))) \
    .withColumn("is_current", F.lit(True)) \
    .withColumn("silver_load_date", F.current_timestamp()) \
    # 5. Limpieza Final: campos sensibles e innecesarios
    .drop("card_number", "cvv", "_rescued_data")
)

In [0]:
# ===================== CARGA A SILVER (STREAMING + MERGE SCD2 PARA CARDS) =====================

# 1. Definimos la función que se ejecutará por cada paquete de datos nuevos
def upsert_cards(df_batch, batch_id):
    # Si no hay datos nuevos en este micro-batch, salimos
    if df_batch.isEmpty():
        return
        
    print(f"Procesando lote {batch_id} de tarjetas...")
    
    # Creamos la vista temporal con el lote actual
    df_batch.createOrReplaceTempView("stg_cards")

    # Si la tabla destino no existe, hacemos la carga inicial (Overwrite)
    if not spark.catalog.tableExists(tabla_silver_cards):
        print(f"La tabla {tabla_silver_cards} no existe. Creando carga inicial...")
        (df_batch.write
         .format("delta")
         .mode("overwrite")
         .saveAsTable(tabla_silver_cards))
        print("Carga inicial de tarjetas completada.")
    else:
        # Si la tabla ya existe, aplicamos el MERGE evaluando los cambios clave
        spark.sql(f"""
        MERGE INTO {tabla_silver_cards} AS target
        USING (
            -- BLOQUE A: Inserciones nuevas o registros actualizados
            SELECT id AS merge_key, stg.* 
            FROM stg_cards stg
            
            UNION ALL
            
            -- BLOQUE B: Generación de la fila clonada para cerrar el registro antiguo
            SELECT NULL AS merge_key, stg.*
            FROM stg_cards stg
            JOIN {tabla_silver_cards} tgt 
              ON stg.id = tgt.id 
              AND tgt.is_current = true
            -- CONDICIONES DE CAMBIO (Null-Safe): 
            -- Vigila límite de crédito, expiración y estado en la dark web
            WHERE NOT (stg.credit_limit <=> tgt.credit_limit)
               OR NOT (stg.expires <=> tgt.expires)
               OR NOT (stg.card_on_dark_web <=> tgt.card_on_dark_web)
        ) AS staged_updates
        ON target.id = staged_updates.merge_key

        -- ACCIÓN 1: CERRAR EL REGISTRO ANTIGUO
        WHEN MATCHED AND target.is_current = true 
          AND (NOT (target.credit_limit <=> staged_updates.credit_limit)
               OR NOT (target.expires <=> staged_updates.expires)
               OR NOT (target.card_on_dark_web <=> staged_updates.card_on_dark_web)) THEN 
          UPDATE SET 
            target.is_current = false, 
            target.valid_to = current_timestamp()

        -- ACCIÓN 2: INSERTAR EL NUEVO REGISTRO
        WHEN NOT MATCHED THEN 
          INSERT *
        """)
        print(f"Merge SCD2 del lote {batch_id} completado.")

# 2. Lanzamos el proceso de Streaming listo para el Workflow
query = (df_silver_cards.writeStream
         .foreachBatch(upsert_cards)
         .option("checkpointLocation", checkpoint_location_s) # Apunta a _checkpoints/cards
         .trigger(availableNow=True) # Se apaga solo cuando termina de procesar todo lo nuevo
         .start())